# 🎭 Emotion Analysis of Letterboxd Reviews

In this notebook, we extend our sentiment analysis pipeline by performing **Emotion Profiling** on the extracted sentences. Instead of a basic positive/negative polarity, we will categorize each sentence into one of seven emotional states: *anger, disgust, fear, joy, neutral, sadness, or surprise*.

We use the `j-hartmann/emotion-english-distilroberta-base` model from Hugging Face for this task.

In [1]:
import os
import torch
import pandas as pd
import plotly.express as px
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import Dataset
from transformers.pipelines.pt_utils import KeyDataset

### 1. Load Data
We load the output from the sentiment analysis step.

In [6]:
# Google Colab Drive Mount & Checkpoint Configuration
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE_DIR = '/content/drive/MyDrive/Projects/letterboxd'
    print("Running in Google Colab. Drive mounted successfully.")
except ImportError:
    print("Not running in Google Colab. Using local directory for checkpoints.")

input_path = os.path.join(DRIVE_BASE_DIR,"letterboxd_sentences_with_sentiment.parquet")
df_sentences = pd.read_parquet(input_path)
print(f"Loaded {len(df_sentences)} sentences.")
df_sentences.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Google Colab. Drive mounted successfully.
Loaded 399909 sentences.


,review_id,film_id,film_name,genres,star_rating_num,sentence_id,sentence_text,is_truncated,language,sentiment_label,sentiment_score,sentiment_numeric
0,1,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,1_0,my favourite part is how he doesn't give an ab...,False,en-US,5 stars,0.616550,5
1,2,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,2_0,tell me you wouldn't cry too if your son grows...,False,en-US,5 stars,0.254817,5
2,3,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,3_0,"I have a headache, but it's the best headache ...",False,en-US,5 stars,0.288259,5
3,4,1,Interstellar,"Adventure,Drama,Science Fiction",4.0,4_0,watched on my 13 inch macbook air just as chri...,False,en-US,5 stars,0.359607,5
4,5,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,5_0,""" It was you.",False,en-US,5 stars,0.301640,5


### 2. Initialize Emotion Model
We initialize the Hugging Face model and pipeline. We use GPU if available.

In [7]:
model_name = 'j-hartmann/emotion-english-distilroberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = 0 if torch.cuda.is_available() else -1
emotion_classifier = pipeline(
    "text-classification", 
    model=model, 
    tokenizer=tokenizer, 
    truncation=True, 
    max_length=512,
    device=device
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

### 3. Run Inference
We use `KeyDataset` to efficiently process the texts in batches.

In [8]:
# Convert pandas dataframe to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df_sentences[['sentence_text']])

BATCH_SIZE = 64
results = []

# Note: For testing purposes, we can limit the dataset (e.g., hf_dataset.select(range(1000)))
for out in tqdm(emotion_classifier(KeyDataset(hf_dataset, 'sentence_text'), batch_size=BATCH_SIZE, truncation=True, max_length=512), total=len(hf_dataset)):
    results.append(out['label'])

df_sentences['predicted_emotion'] = results
df_sentences.head()

  0%|          | 0/399909 [00:00<?, ?it/s]

KeyboardInterrupt: 

### 4. Movie-Level Aggregation
We aggregate the emotions per movie to generate percentage distributions.

In [ ]:
emotions = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

# Count instances of each emotion per film
emotion_counts = df_sentences.groupby(['film_id', 'film_name', 'predicted_emotion']).size().unstack(fill_value=0)

# Normalize to percentages
emotion_pct = emotion_counts.div(emotion_counts.sum(axis=1), axis=0)
emotion_pct.columns = [f"emotion_{col}" for col in emotion_pct.columns]

# Ensure all emotions have columns
for emo in emotions:
    col_name = f"emotion_{emo}"
    if col_name not in emotion_pct.columns:
        emotion_pct[col_name] = 0.0

emotion_pct = emotion_pct.reset_index()
emotion_pct.head()

### 5. Visualization: Emotional Signature Radar Chart
Let's visualize the emotional profile for a sample movie.

In [ ]:
# Select a random movie with enough data
sample_movie = emotion_pct.iloc[0]

emotion_cols = [f"emotion_{e}" for e in emotions]
emotion_values = [sample_movie[col] for col in emotion_cols]

radar_df = pd.DataFrame({
    'Emotion': [e.capitalize() for e in emotions],
    'Score': emotion_values
})

fig = px.line_polar(
    radar_df, 
    r='Score', 
    theta='Emotion', 
    line_close=True,
    title=f"Emotional Signature: {sample_movie['film_name']}"
)
fig.update_traces(fill='toself', line_color='#00E054')
fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, max(0.5, max(emotion_values) + 0.1)])
    ),
    showlegend=False
)
fig.show()

### 6. Save Data
Finally, we can save the sentence-level data to our processed folder.

In [ ]:
output_path = "letterboxd_sentences_emotion.parquet"
df_sentences.to_parquet(output_path, index=False)
print(f"Saved emotion-analyzed sentences to {output_path}")